# 🟢 Grammora AI — All-in-One Urdu/English Instruction Model · **single cell**

Run the one cell below top to bottom. It fine-tunes **Qwen2.5** into **one** model that does
grammar, paraphrasing, summarization, translation (both ways), Q&A and text generation.

**This version:** uses your **whole** dataset (no 500k cap on QA), **removes duplicate
records**, and **keeps only Urdu/English** rows — nothing wasted, nothing approximated.
Just set `DATA_DIR` at the top of the cell.

In [ ]:
# =============================================================================
# GRAMMORA AI — ALL-IN-ONE, SINGLE CELL, PRODUCTION
# grammar · paraphrasing · summarization · translation(both ways) · QA · text-gen
# Whole dataset · dedup · Urdu/English-only · completion-only masking · LoRA
# =============================================================================

# ---- 0. EDIT THESE ----------------------------------------------------------
DATA_DIR   = "/workspace/jsonl_datasets"      # folder with your 7 .jsonl files
OUT_DIR    = "/workspace/grammora_out"        # checkpoints + final model
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"       # 3B for fast test, 14B for higher ceiling
MAX_STEPS  = 30000                            # raise to see more of the corpus (see notes)
DO_MERGE   = True                             # merge LoRA -> standalone model at the end

# ---- 1. install deps (safe to re-run) --------------------------------------
import importlib.util, subprocess, sys
_req = [("transformers","transformers>=4.44"),("datasets","datasets>=2.20"),
        ("peft","peft>=0.12"),("accelerate","accelerate>=0.33"),
        ("sentencepiece","sentencepiece"),("sacrebleu","sacrebleu")]
_missing = [pip for mod,pip in _req if importlib.util.find_spec(mod) is None]
if _missing:
    print("installing:", _missing)
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*_missing])
    importlib.invalidate_caches()

import os, json, math, random, time, gc, hashlib
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, TrainingArguments,
                          Trainer, TrainerCallback, set_seed)
from datasets import IterableDataset, Features, Value
from peft import LoraConfig, get_peft_model, PeftModel
import sacrebleu

# ---- 2. config --------------------------------------------------------------
@dataclass
class Config:
    data_dir: str = DATA_DIR
    out_dir: str = OUT_DIR
    base_model: str = BASE_MODEL
    max_seq_len: int = 2048
    train_mode: str = "lora"          # "lora" or "qlora"
    lora_r: int = 64; lora_alpha: int = 128; lora_dropout: float = 0.05
    lora_target_modules: Tuple[str,...] = ("q_proj","k_proj","v_proj","o_proj",
                                           "gate_proj","up_proj","down_proj")
    # data rules
    translation_bidirectional: bool = True   # en->ur AND ur->en
    include_urdu_corpus_lm: bool = True
    add_system_prompt: bool = True
    dedup: bool = True                       # drop exact-duplicate examples
    lang_filter: bool = True                 # keep only Urdu/English rows
    lang_max_other: float = 0.05             # max fraction of foreign-script letters
    eval_holdout_per_file: int = 150         # first N rows/file reserved for eval
    # optimization
    micro_batch_size: int = 8; grad_accum_steps: int = 8
    max_steps: int = MAX_STEPS
    learning_rate: float = 1e-4; warmup_ratio: float = 0.03
    weight_decay: float = 0.0; grad_clip: float = 1.0; lr_scheduler: str = "cosine"
    bf16: bool = True; gradient_checkpointing: bool = True
    num_workers: int = 4; shuffle_buffer: int = 50000; seed: int = 3407
    logging_steps: int = 10; save_steps: int = 1000; save_total_limit: int = 3
    sample_every: int = 500; eval_every: int = 1000; resume: bool = True
    system_prompt: str = ("آپ Grammora ہیں، ایک اعلیٰ معیار کا اردو اور انگریزی معاون۔ "
        "آپ گرامر کی درستگی، خلاصہ، ترجمہ، سوال و جواب اور تحریر میں مدد کرتے ہیں۔")

cfg = Config()
set_seed(cfg.seed); random.seed(cfg.seed)
os.makedirs(cfg.out_dir, exist_ok=True)
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
cfg.bf16 = cfg.bf16 and BF16_OK
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} {p.total_memory/1024**3:.0f}GB")
def flash_available():
    return importlib.util.find_spec("flash_attn") is not None

# ---- 3. language filter (keep Urdu/English only) ---------------------------
def _script_counts(s):
    ar = lat = oth = 0
    for ch in s:
        if not ch.isalpha():         # skip digits, punctuation, spaces
            continue
        o = ord(ch)
        if 0x0600 <= o <= 0x06FF or 0x0750 <= o <= 0x077F or \
           0xFB50 <= o <= 0xFDFF or 0xFE70 <= o <= 0xFEFF:
            ar += 1                  # Arabic/Urdu script
        elif (0x41 <= o <= 0x5A) or (0x61 <= o <= 0x7A):
            lat += 1                 # basic Latin (English)
        else:
            oth += 1                 # anything else (Hindi/Chinese/etc.)
    return ar, lat, oth

def is_ur_or_en(text):
    if not cfg.lang_filter:
        return True
    ar, lat, oth = _script_counts(text)
    tot = ar + lat + oth
    if tot < 1:
        return False                 # no real letters -> junk
    return (oth / tot) <= cfg.lang_max_other

# ---- 4. record normalization (incl. bidirectional translation) -------------
FILE_SPECS = {"grammar.jsonl":"chat","paraphrasing.jsonl":"chat",
    "summarization.jsonl":"chat","text_generation.jsonl":"chat",
    "question_answering.jsonl":"chat","translation.jsonl":"translation",
    "urdu_corpus.jsonl":"text"}
_UR2EN = ["Translate this into English.","Translate the following Urdu text to English.",
          "Convert this Urdu passage into English.","Render this in English."]
_EN2UR = ["Translate this into Urdu.","Translate the following English text to Urdu.",
          "اس انگریزی متن کا اردو ترجمہ کریں۔","Convert this English passage into Urdu."]

def _split_instr(uc):
    if "\n\n" in uc:
        a, b = uc.split("\n\n", 1); return a.strip(), b.strip()
    return "", uc.strip()

def normalize_record(rec, spec, rng):
    if spec == "text":
        t = (rec.get("text") or "").strip()
        return [{"kind":"text","messages":[],"text":t}] if t else []
    msgs = rec.get("messages") or []
    if not msgs: return []
    if spec == "translation":
        user = next((m for m in msgs if m.get("role")=="user"), None)
        asst = next((m for m in msgs if m.get("role")=="assistant"), None)
        if not user or not asst: return []
        _, english = _split_instr(user.get("content",""))
        urdu = (asst.get("content") or "").strip()
        if not english or not urdu: return []
        out = [{"kind":"chat","text":"","messages":[
            {"role":"user","content":rng.choice(_EN2UR)+"\n\n"+english},
            {"role":"assistant","content":urdu}]}]
        if cfg.translation_bidirectional:
            out.append({"kind":"chat","text":"","messages":[
                {"role":"user","content":rng.choice(_UR2EN)+"\n\n"+urdu},
                {"role":"assistant","content":english}]})
        return out
    clean = [{"role":m.get("role"),"content":(m.get("content") or "").strip()}
             for m in msgs if (m.get("content") or "").strip()]
    if not any(m["role"]=="assistant" for m in clean): return []
    return [{"kind":"chat","text":"","messages":clean}]

def _example_text(ex):
    if ex["kind"] == "text": return ex["text"]
    return "  ".join(m["role"]+":"+m["content"] for m in ex["messages"])

def _fingerprint(ex):
    return hashlib.blake2b(_example_text(ex).encode("utf-8"), digest_size=8).digest()

# ---- 5. combined stream: ALL files, round-robin, dedup + language filter ----
# Uses every record once (no cap). Round-robin interleaves tasks. Dedup set and
# language filter shrink the corpus without approximating it.
def combined_stream(skip_holdout=True):
    seen = set()
    files = [(f,s,os.path.join(cfg.data_dir,f)) for f,s in FILE_SPECS.items()
             if os.path.exists(os.path.join(cfg.data_dir,f))
             and not (s=="text" and not cfg.include_urdu_corpus_lm)]
    handles = {f: open(p,"r",encoding="utf-8") for f,s,p in files}
    specs = {f:s for f,s,p in files}
    rng = random.Random(cfg.seed)
    skip_left = {f: (cfg.eval_holdout_per_file if skip_holdout else 0) for f in handles}
    alive = set(handles)
    kept = dropped_lang = dropped_dup = 0
    while alive:
        for f in list(handles):
            if f not in alive: continue
            line = handles[f].readline()
            if not line:
                alive.discard(f); handles[f].close(); continue
            if skip_left[f] > 0:
                skip_left[f] -= 1; continue
            line = line.strip()
            if not line: continue
            try: rec = json.loads(line)
            except Exception: continue
            for ex in normalize_record(rec, specs[f], rng):
                if not is_ur_or_en(_example_text(ex)):
                    dropped_lang += 1; continue
                if cfg.dedup:
                    fp = _fingerprint(ex)
                    if fp in seen: dropped_dup += 1; continue
                    seen.add(fp)
                kept += 1
                if kept % 500000 == 0:
                    print(f"  stream: kept={kept:,} dropped_dup={dropped_dup:,} "
                          f"dropped_lang={dropped_lang:,}")
                yield ex

# ---- 6. tokenizer + model --------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token
print("tokenizer vocab:", len(tokenizer))

_dtype = torch.bfloat16 if cfg.bf16 else torch.float16
_quant = None
if cfg.train_mode == "qlora":
    from transformers import BitsAndBytesConfig
    _quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=_dtype, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model, torch_dtype=_dtype, quantization_config=_quant,
    attn_implementation="flash_attention_2" if flash_available() else "sdpa",
    trust_remote_code=True)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
if cfg.train_mode == "qlora":
    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=cfg.gradient_checkpointing)
model = get_peft_model(model, LoraConfig(
    r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
    target_modules=list(cfg.lora_target_modules), bias="none", task_type="CAUSAL_LM"))
model.print_trainable_parameters()
if cfg.gradient_checkpointing:
    if hasattr(model,"enable_input_require_grads"): model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# ---- 7. tokenize + completion-only masking + collator ----------------------
def make_tok_fn():
    L = cfg.max_seq_len; eos = tokenizer.eos_token_id
    def empty(): return {"input_ids":[], "labels":[], "attention_mask":[]}
    def fn(ex):
        if ex["kind"] == "text":
            t = (ex.get("text") or "").strip()
            if not t: return empty()
            ids = tokenizer(t, add_special_tokens=False, truncation=True,
                            max_length=L-1)["input_ids"] + [eos]
            return {"input_ids":ids,"labels":list(ids),"attention_mask":[1]*len(ids)}
        msgs = ex.get("messages") or []
        if cfg.add_system_prompt and not any(m["role"]=="system" for m in msgs):
            msgs = [{"role":"system","content":cfg.system_prompt}] + msgs
        if not msgs or msgs[-1]["role"] != "assistant": return empty()
        full = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=False)
        prm  = tokenizer.apply_chat_template(msgs[:-1], tokenize=True, add_generation_prompt=True)
        full = full[:L]; n = min(len(prm), len(full))
        labels = ([-100]*n + list(full[n:]))[:len(full)]
        if all(x==-100 for x in labels): return empty()
        return {"input_ids":full,"labels":labels,"attention_mask":[1]*len(full)}
    return fn

class PadCollator:
    def __init__(self, pad, mult=8): self.pad=pad; self.mult=mult
    def __call__(self, batch):
        m = max(len(b["input_ids"]) for b in batch)
        m = ((m + self.mult - 1)//self.mult)*self.mult
        ids=lbl=att=None; I=[];Lb=[];A=[]
        for b in batch:
            k = m - len(b["input_ids"])
            I.append(b["input_ids"]+[self.pad]*k)
            Lb.append(b["labels"]+[-100]*k)
            A.append(b["attention_mask"]+[0]*k)
        return {"input_ids":torch.tensor(I),"labels":torch.tensor(Lb),
                "attention_mask":torch.tensor(A)}

_features = Features({"kind":Value("string"),
    "messages":[{"role":Value("string"),"content":Value("string")}],"text":Value("string")})
train_ds = IterableDataset.from_generator(combined_stream, features=_features)
train_ds = train_ds.shuffle(seed=cfg.seed, buffer_size=cfg.shuffle_buffer)
train_ds = train_ds.map(make_tok_fn(), remove_columns=["kind","messages","text"])
train_ds = train_ds.filter(lambda e: len(e["input_ids"]) > 0)
collator = PadCollator(tokenizer.pad_token_id)

# ---- 8. held-out eval (first N rows/file, disjoint from training) ----------
def build_eval(per_task=40):
    ev = {}; rng = random.Random(cfg.seed)
    for f, spec in FILE_SPECS.items():
        p = os.path.join(cfg.data_dir, f)
        if not os.path.exists(p): continue
        rows = []
        with open(p,"r",encoding="utf-8") as fh:
            for line in fh:
                if len(rows) >= per_task: break
                line=line.strip()
                if not line: continue
                try: rec=json.loads(line)
                except Exception: continue
                for ex in normalize_record(rec, spec, rng):
                    if is_ur_or_en(_example_text(ex)): rows.append(ex)
        ev[f.replace(".jsonl","")] = rows[:per_task]
    return ev
eval_examples = build_eval()
print("eval per task:", {k:len(v) for k,v in eval_examples.items()})

@torch.no_grad()
def eval_loss_ppl():
    model.eval(); tf = make_tok_fn(); tl=tt=0; dev=next(model.parameters()).device
    for rows in eval_examples.values():
        for ex in rows:
            enc = tf(ex)
            if not enc["input_ids"]: continue
            ids=torch.tensor([enc["input_ids"]],device=dev)
            lb=torch.tensor([enc["labels"]],device=dev)
            out=model(input_ids=ids, labels=lb); n=int((lb!=-100).sum())
            tl+=float(out.loss)*n; tt+=n
    model.train(); loss=tl/max(tt,1); return loss, math.exp(min(loss,20))

@torch.no_grad()
def eval_chrf(n=30):
    if "translation" not in eval_examples: return None
    model.eval(); dev=next(model.parameters()).device; H=[];R=[]
    for ex in eval_examples["translation"][:n]:
        m0=ex["messages"][0]
        pr=([{"role":"system","content":cfg.system_prompt}]+[m0]) if cfg.add_system_prompt else [m0]
        ids=tokenizer.apply_chat_template(pr,tokenize=True,add_generation_prompt=True,return_tensors="pt").to(dev)
        o=model.generate(ids,max_new_tokens=200,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        H.append(tokenizer.decode(o[0,ids.shape[1]:],skip_special_tokens=True).strip())
        R.append(ex["messages"][1]["content"])
    model.train(); return sacrebleu.corpus_chrf(H,[R]).score

EVAL_PROMPTS=[
 ("grammar","اس کی گرامر درست کریں۔\n\nمیں کل اسکول جاتا ہوں اور کتاب پڑھی تھی۔"),
 ("summarize","خلاصہ کریں۔\n\nپاکستان اسٹاک مارکیٹ میں آج زبردست تیزی دیکھی گئی اور انڈیکس چار سو پوائنٹس بڑھ کر بند ہوا۔"),
 ("translate_en_ur","Translate this into Urdu.\n\nEducation is the most powerful weapon to change the world."),
 ("translate_ur_en","Translate this into English.\n\nعلم حاصل کرنا ہر مرد اور عورت پر فرض ہے۔"),
 ("qa","سوال کا جواب دیں۔\n\nپاکستان کا دارالحکومت کون سا شہر ہے؟"),
 ("write","Write a detailed article for the headline:\n\nThe importance of clean drinking water."),
]
@torch.no_grad()
def show_samples(mx=150):
    model.eval(); dev=next(model.parameters()).device
    print("-- LIVE SAMPLES " + "-"*40)
    for task, p in EVAL_PROMPTS:
        msgs=([{"role":"system","content":cfg.system_prompt}] if cfg.add_system_prompt else [])+[{"role":"user","content":p}]
        ids=tokenizer.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,return_tensors="pt").to(dev)
        o=model.generate(ids,max_new_tokens=mx,do_sample=True,temperature=0.7,top_p=0.9,
                         repetition_penalty=1.1,pad_token_id=tokenizer.pad_token_id)
        print(f"  [{task}] " + tokenizer.decode(o[0,ids.shape[1]:],skip_special_tokens=True).strip()[:300])
    model.train()

# ---- 9. trainer + live monitoring ------------------------------------------
class Monitor(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs: logs["ppl"]=round(math.exp(min(logs["loss"],20)),2)
    def on_step_end(self, args, state, control, model=None, **kw):
        s=state.global_step
        if cfg.sample_every and s>0 and s%cfg.sample_every==0:
            try: show_samples()
            except Exception as e: print("sample skip:",e)
        if cfg.eval_every and s>0 and s%cfg.eval_every==0:
            try:
                l,pp=eval_loss_ppl(); cf=eval_chrf()
                print(f"[eval @ {s}] loss={l:.4f} ppl={pp:.2f}" + (f" chrF={cf:.1f}" if cf else ""))
            except Exception as e: print("eval skip:",e)
            gc.collect(); torch.cuda.empty_cache()

args = TrainingArguments(
    output_dir=cfg.out_dir, per_device_train_batch_size=cfg.micro_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps, max_steps=cfg.max_steps,
    learning_rate=cfg.learning_rate, lr_scheduler_type=cfg.lr_scheduler,
    warmup_ratio=cfg.warmup_ratio, weight_decay=cfg.weight_decay, max_grad_norm=cfg.grad_clip,
    bf16=cfg.bf16, fp16=not cfg.bf16, logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps, save_total_limit=cfg.save_total_limit,
    gradient_checkpointing=cfg.gradient_checkpointing,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=cfg.num_workers, dataloader_pin_memory=True,
    report_to="none", remove_unused_columns=False,
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch", seed=cfg.seed)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=collator, callbacks=[Monitor()])

# ---- 10. baseline -> train -> final eval -----------------------------------
try:
    l,pp=eval_loss_ppl(); cf=eval_chrf()
    print(f"BEFORE  loss={l:.4f} ppl={pp:.2f}" + (f" chrF={cf:.1f}" if cf else ""))
except Exception as e: print("baseline skip:",e)

def _newest():
    if not cfg.resume: return None
    ck=sorted(Path(cfg.out_dir).glob("checkpoint-*"),
              key=lambda p:int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else 0)
    return str(ck[-1]) if ck else None
_resume=_newest(); print("resume from:", _resume or "scratch")
trainer.train(resume_from_checkpoint=_resume)

final=os.path.join(cfg.out_dir,"final"); trainer.save_model(final); tokenizer.save_pretrained(final)
with open(os.path.join(final,"grammora_config.json"),"w",encoding="utf-8") as f:
    json.dump(asdict(cfg), f, ensure_ascii=False, indent=2)
print("adapter saved ->", final)
try:
    l,pp=eval_loss_ppl(); cf=eval_chrf()
    print(f"AFTER   loss={l:.4f} ppl={pp:.2f}" + (f" chrF={cf:.1f}" if cf else ""))
except Exception as e: print("final eval skip:",e)
show_samples(mx=220)

# ---- 11. quick chat demo (in-memory model) ---------------------------------
@torch.no_grad()
def grammora(prompt, mx=400, temperature=0.7):
    model.eval(); dev=next(model.parameters()).device
    msgs=([{"role":"system","content":cfg.system_prompt}] if cfg.add_system_prompt else [])+[{"role":"user","content":prompt}]
    ids=tokenizer.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,return_tensors="pt").to(dev)
    o=model.generate(ids,max_new_tokens=mx,do_sample=True,temperature=temperature,top_p=0.9,
                     repetition_penalty=1.1,pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(o[0,ids.shape[1]:],skip_special_tokens=True).strip()
for _p in ["اس کی گرامر درست کریں۔ میں کل اسکول جاتا ہوں اور کتاب پڑھی تھی۔",
           "Translate this into English. علم روشنی کی مانند ہے۔",
           "Summarize this. پاکستان اسٹاک مارکیٹ میں آج زبردست تیزی دیکھی گئی۔"]:
    print("Q:", _p); print("A:", grammora(_p), "\n")

# ---- 12. merge LoRA -> standalone model ------------------------------------
if DO_MERGE:
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    merged=os.path.join(cfg.out_dir,"merged")
    base=AutoModelForCausalLM.from_pretrained(cfg.base_model,torch_dtype=torch.bfloat16,trust_remote_code=True)
    mm=PeftModel.from_pretrained(base, final).merge_and_unload()
    mm.config.use_cache=True
    mm.save_pretrained(merged, safe_serialization=True); tokenizer.save_pretrained(merged)
    print("standalone model ->", merged)
    print("serve: python -m vllm.entrypoints.openai.api_server --model", merged)
print("DONE ✅")

### Notes
- **Whole dataset, no cap:** every valid record of every file is streamed once
  (round-robin so tasks stay mixed). Duplicates are removed and non-Urdu/English rows
  dropped, so nothing is wasted and nothing is approximated.
- **QA (~5 crore):** kept in full; its repeated answers across question variants are
  distinct (question, answer) pairs, so dedup keeps the useful variety and only drops
  exact repeats. First run streams the whole 84 GB file, so the dedup set uses a few GB
  of RAM (you have plenty) and the first pass is I/O-heavy.
- **`MAX_STEPS`** bounds *compute*, not data. One pass over the full corpus is enormous;
  watch the `[eval]` loss/ppl and `chrF` lines and raise `MAX_STEPS` until they plateau.
- **Multi-GPU:** a notebook cell is one process (one GPU). To use both RTX PRO 6000s,
  run the `grammora_sft.py` script with `accelerate launch --multi_gpu --num_processes 2`.